# Example 04 - 2D FES to Multi-Basin MFEP CTMC

This notebook mirrors `examples/04_mfep_ctmc.py` using the bundled
synthetic 2D FES. It detects all basins, computes pairwise MFEP legs,
assembles the CTMC rate matrix, and visualizes the full network.

This is one of the heavier examples because each basin pair requires
an MFEP and 1D arc-length reduction.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stochkin as sk

from stochkin.plotting import _apply_pub_axes, _apply_pub_cbar
from stochkin.style import publication_style


In [ ]:
fes2d_path = DATA_DIR / "synthetic_2d_fes.dat"
D_s = 0.05
T = 300.0
neb_images = 120
neb_steps = 8000
max_basins = None
core_fraction = 0.05

result = sk.run_multi_mfep_ctmc(
    fes2d_path=fes2d_path,
    D_s=D_s,
    T=T,
    neb_images=neb_images,
    neb_steps=neb_steps,
    max_basins=max_basins,
    core_fraction=core_fraction,
)


In [ ]:
basin_labels = [f"B{bid}" for bid in result["basin_ids"]]
K_ns = pd.DataFrame(
    result["K_ps"] * 1000.0,
    index=basin_labels,
    columns=basin_labels,
)
exit_times_ns = pd.Series(
    result["exit_ps"] / 1000.0,
    index=basin_labels,
    name="exit_time_ns",
)

print(f"Detected {len(basin_labels)} basins")
for basin in result["basin_network"].basins:
    print(f"Basin {basin.id}: minimum {basin.minimum}, F_min = {basin.f_min:.2f} kJ/mol")


In [ ]:
K_ns


In [ ]:
exit_times_ns


In [ ]:
x_grid, y_grid, fes_grid = sk.load_plumed_fes_2d(fes2d_path, verbose=False)
basin_net = result["basin_network"]
kT = result["kT"]

out_fes = OUT_DIR / "04_basins_on_fes.png"
out_rates = OUT_DIR / "04_rate_matrix.png"
out_exit = OUT_DIR / "04_exit_times.png"

with publication_style():
    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    cs = ax.contourf(
        x_grid,
        y_grid,
        (fes_grid / kT).T,
        levels=np.linspace(0, 15, 30),
        cmap="rainbow_r",
    )
    cbar = fig.colorbar(cs, ax=ax)
    _apply_pub_cbar(cbar, label=r"$F / k_\mathrm{B}T$")

    for basin in basin_net.basins:
        ax.plot(*basin.minimum, "o", color="white", ms=10, zorder=5)
        ax.text(
            basin.minimum[0],
            basin.minimum[1],
            f" B{basin.id}",
            color="white",
            va="center",
            fontsize=9,
            fontweight="bold",
            zorder=6,
        )

    for (_, _), leg in result["legs"].items():
        path = leg["mfep_path"]
        ax.plot(path.x, path.y, "--", color="white", lw=1.5, alpha=0.8)

    _apply_pub_axes(ax, r"CV$_1$", r"CV$_2$", "Basins + MFEPs on 2D FES")
    fig.tight_layout()
    fig.savefig(out_fes, dpi=300)

with publication_style():
    K_abs = np.abs(K_ns.to_numpy())
    with np.errstate(divide="ignore", invalid="ignore"):
        Klog = np.where(K_abs > 0, np.log10(K_abs), np.nan)

    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    im = ax.imshow(Klog, cmap="magma_r", aspect="auto")
    ax.set_xticks(range(len(basin_labels)))
    ax.set_xticklabels(basin_labels)
    ax.set_yticks(range(len(basin_labels)))
    ax.set_yticklabels(basin_labels)
    cbar = fig.colorbar(im, ax=ax)
    _apply_pub_cbar(cbar, label=r"$\log_{10}|K_{ij}|$  [ns$^{-1}$]")
    _apply_pub_axes(ax, title=f"Rate matrix ({len(basin_labels)} basins)")
    fig.tight_layout()
    fig.savefig(out_rates, dpi=300)

with publication_style():
    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    bars = ax.bar(range(len(basin_labels)), exit_times_ns.to_numpy(), color="steelblue")
    ax.set_yscale("log")
    ax.set_xticks(range(len(basin_labels)))
    ax.set_xticklabels(basin_labels)
    for bar, value in zip(bars, exit_times_ns.to_numpy()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    _apply_pub_axes(ax, xlabel="Basin", ylabel="Exit time  [ns]", title="Mean exit times")
    fig.tight_layout()
    fig.savefig(out_exit, dpi=300)

print(f"Saved {out_fes.relative_to(ROOT)}")
print(f"Saved {out_rates.relative_to(ROOT)}")
print(f"Saved {out_exit.relative_to(ROOT)}")
plt.show()
